# Freeze factor-recoverability and statistics supplement
Attach only private derived dataset `thestonedape/task-aware-eegtotext`, enable Internet, and enable private secret `GITHUB_TOKEN`. This CPU-only notebook freezes train/validation factor rows, subject-held-out exclusions, cross-task duplicate identities, probe conditions, metrics, seeds, and admission rules. It does not load EEG arrays, fit probes, or access test rows.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'f66330c7e50f0e60297211e8823071b69e7d2b99'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-recoverability-protocol'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
assert len(COMMIT) == 40 and len(EXPECTED_INDEX_SHA256) == 64

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
print({'python': platform.python_version(), 'platform': platform.platform()})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == COMMIT
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_protocol_manifests.py')], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_recoverability_protocol.py')], check=True)

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
subprocess.run([
    sys.executable, os.path.join(WORKTREE, 'evaluation', 'build_recoverability_protocol.py'),
    '--dataset-root', dataset_root, '--output-root', OUTPUT,
    '--expected-index-sha256', EXPECTED_INDEX_SHA256,
], check=True)

In [ ]:
report_path = os.path.join(OUTPUT, 'recoverability_contract_report.json')
report = json.load(open(report_path, encoding='utf-8'))
registry = json.load(open(os.path.join(OUTPUT, 'recoverability_registry.json'), encoding='utf-8'))
assert report['status'] == 'pass'
assert report['source']['index_sha256'] == EXPECTED_INDEX_SHA256
assert report['counts']['train_rows'] == 17908 and report['counts']['validation_rows'] == 2200
assert report['checks']['held_out_test_accessed'] is False
assert report['checks']['missing_semkey_labels_fabricated'] is False
assert report['checks']['frozen_glim_representation_called_eeg_only'] is False
assert report['checks']['ordinary_factor_probes_fit_viable'] is True
assert report['checks']['subject_fold_training_exclusions_nonempty'] is True
assert registry['held_out_test_accessed'] is False
assert registry['unavailable_factors']['gpt2_mean_nll_v1'] == 'not_fabricated'
assert registry['duplicate_consistency']['validation_group_count'] == 0
for name, expected in report['artifact_sha256'].items():
    state = hashlib.sha256()
    with open(os.path.join(OUTPUT, name), 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    assert state.hexdigest() == expected, name
run_metadata = {
    'status': 'pass', 'project_commit': actual_commit, 'python': platform.python_version(),
    'dataset_index_sha256': EXPECTED_INDEX_SHA256, 'test_accessed': False,
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
print({'dataset_root': dataset_root, **report['counts'], 'checks': report['checks'], 'duplicate_consistency_status': report['duplicate_consistency_status']})
shutil.rmtree(WORKTREE)
print('FROZEN RECOVERABILITY PROTOCOL SUPPLEMENT: PASS')